---
---

## Introduction

In this first laboratory we will see a few ways to exploit and adapt pre-trained models to solve new problems. We will start first by downloading and instantiating a new dataset and establishing a stable and reproducible baseline model based on a pre-trained CNN.

---

## Exercise 1 (Warmup): Exploratory Data Analysis and a Stable Baseline

For this laboratory we will work with [The German Traffic Sign Detection Benchmark](https://benchmark.ini.rub.de/) [1]. We will begin, not with *detection*, but with a simpler traffic sign *classification* problem. This has two advantages: (1) the images are *smaller* than in the detection benchmark; and (2) a wrapper for the GTSRB dataset is conveniently included in the `torchvision` library.  

[1] Houben S, Stallkamp J, Salmen J, Schlipsing M, Igel C. Detection of traffic signs in real-world images: The German Traffic Sign Detection Benchmark. In The 2013 International Joint Conference on Neural Networks (IJCNN), 2013.

### Exercise 1.1: Exploratory Data Analysis

A good best practice to adopt in all experimental deep learning projects is thorough *exploratory data analysis*. In this exercise you should instantiate the GTSRB Dataset, inspect some images, and do some statistical analysis of the distribution of data (and metadata). A goal here is to keep an eye out for anything that might be problematic in what is to come.



In [ ]:
# Standard imports and aliases.
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision.transforms.v2 as T  
import torchvision.transforms.functional as tf
import pandas as pd
import torch.nn.functional as F
import torch.nn as nn


from torchvision.datasets import GTSRB
from torchvision.models import get_model
from torch.utils.data import DataLoader, random_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from omegaconf import OmegaConf
from datasets import load_dataset
from torchvision.models.detection.faster_rcnn import _default_anchorgen, FastRCNNConvFCHead, RPNHead, FasterRCNN
from torchvision.models.detection.backbone_utils import _resnet_fpn_extractor
from torchvision.ops import box_convert
from torchvision.utils import draw_bounding_boxes
from tqdm import tqdm
from torchmetrics.detection import MeanAveragePrecision

In [ ]:
transform = T.Compose([T.ToTensor()])
ds_train = GTSRB('_data/', split='train', transform=transform, download=True)
ds_test  = GTSRB('_data/', split='test', transform=transform, download=True)

In [ ]:
num_samples = 100
image_samples = [ds_train[idx] for idx in np.random.choice(range(len(ds_train)), num_samples)]

plt.figure(figsize=(15,20))
for (idx, (im, cls)) in enumerate(image_samples):
    plt.subplot(10, 10, idx+1)
    plt.imshow(im.permute([1, 2, 0]))
    plt.title(f'CLS: {cls}')
    plt.axis('off')

In [ ]:
df_stats = pd.DataFrame([(cls, im.shape[1], im.shape[2]) for (im, cls) in ds_train], columns=['CLS', 'HEIGHT', 'WIDTH'])
df_stats['AR'] = df_stats['WIDTH'] / df_stats['HEIGHT']
df_stats.describe()

In [ ]:
df_stats.hist()

In [ ]:
transform = T.Compose([T.Resize(70), T.RandomCrop((64, 64)), T.ToTensor(), T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
ds_train = GTSRB('_data/', split='train', transform=transform, download=True)
ds_test  = GTSRB('_data/', split='test', transform=transform, download=True)

In [ ]:
num_samples = 100
image_samples = [ds_train[idx] for idx in np.random.choice(range(len(ds_train)), num_samples)]

plt.figure(figsize=(15,20))
for (idx, (im, cls)) in enumerate(image_samples):
    plt.subplot(10, 10, idx+1)
    plt.imshow(im.permute([1, 2, 0]))
    plt.title(f'CLS: {cls}')
    plt.axis('off')

### Analysis

Deep Learning is very much an *experimental* discipline. Experiments are *nothing* without analysis and interpretation. Be sure to **always** stop and analyze the results of preliminary explorations. Note anything significant and -- importantly -- anything that is going to be relevant for what comes next.

So... In this Markdown cell you should collect and report (using, for the love of God, the *rich markup capabilities of Markdown*) any relevant findings you have made before proceding.

**Important Warning**: This is the **one and only** time I will remind you of the need to provide *analysis* and interpretation of your experimental methodology and results. The responsibility is *yours* to include it elsewhere.


---
### Exercise 1.2: A Stable and Reproducible Baseline

In this exercise you should implement code to use a pretrained network as a *feature extractor* that, instead of *classifying* images in input, should return the *feature representation* from the last layer of the pretrained model before the classifier. These features, extracted from the train set, should be used to train a *classical* model for classification (e.g. an SVM, a Nearest Neighbor, or a Linear Discriminant classifier from Scikit-learn). Evaluate the performance of this baseline model on the features extracted from the test set.

In [ ]:
batch_size = 1024
device = 'cuda' if torch.cuda.is_available() else 'cpu'

dl_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True)
dl_test = DataLoader(ds_test, batch_size=batch_size, shuffle=False)

model = get_model('resnet50', weights='DEFAULT')
model.fc = nn.Identity() # Questo mi serve a fare feature extraction
model = model.to(device)

In [ ]:
def feature_extraction(model, dl):
    feats = []
    classes = []

    model.eval()
    for (ims, cls) in tqdm(dl_train):
        ims = ims.to(device)
        with torch.no_grad():
            output = model(ims)
            feats.append(output)
            classes.append(cls)

    feats = torch.vstack(feats).cpu()
    classes = torch.concat(classes)

    return feats, classes

In [ ]:
train_feats, train_classes = feature_extraction(model=model, dl=dl_train)
test_feats, test_classes = feature_extraction(model=model, dl=dl_test)
model = model.to('cpu')

In [ ]:
svc = SVC(kernel='poly')
svc.fit(train_feats, train_classes)

print(classification_report(svc.predict(test_feats), test_classes))

This simple SVM performs really well: **95%** of accuracy


---
### Exercise 1.3: A Fine-tuning Baseline

In this exercise you should try to *improve* on the stable baseline given by the feature extraction + SVM (or whatever) classifier you produced in the previous lecture. To do this, you should *fine-tune* the ResNet-18 (or whatever model you chose) to solve the new classification task.

To do this, you could proceed by:
1. Loading the ResNet-18 (or whatever) model and replacing the final FC layer (the classifier) with a *new* classifier. This could be a single Linear layer, or could be an MLP.
2. Training the resulting model on the GTSRB dataset for a few epochs.
3. Evaluating the resulting performance.

Some things you should probably consider (especially thinking about the *next* exercise):
+ You should be monitoring not only the loss on the training set, but also a *validation* loss on an *independent* validation set. Split the training set into two datasets: one with, say, 80% of the original training samples, and another with the remaining 20%. You can use this smaller set to monitor performance and check for *overfitting*.
+ Maybe the best strategy is to not fine-tune *all* layers, but only the last few. Think about *selectively* fine-tuning layers of the network.
+ Maybe a *single* linear layer isn't the best option for the classifier. Think about using an MLP instead.

In [ ]:
transform = T.Compose([T.Resize(70), T.RandomCrop((64, 64)), T.ToTensor(), T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
training_set = GTSRB('_data/', split='train', transform=transform, download=True)
training_set_size = int(0.8 * len(training_set))
validation_set_size = len(training_set) - training_set_size

generator = torch.Generator().manual_seed(1302)
training_set, validation_set = random_split(
    training_set,
    [training_set_size, validation_set_size],
    generator=generator
)
test_set  = GTSRB('_data/', split='test', transform=transform, download=True)

In [ ]:
def train_epoch(model, dl, opt, epoch='Unknown', device='cpu'):
    model.train()
    losses = []
    for (xs, ys) in tqdm(dl, desc=f'Training epoch {epoch}', leave=True):
        xs = xs.to(device)
        ys = ys.to(device)
        opt.zero_grad()
        logits = model(xs)
        loss = F.cross_entropy(logits, ys)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    return np.mean(losses)

def evaluate_model(model, dl, device='cpu'):
    model.eval()
    predictions = []
    gts = []
    for (xs, ys) in tqdm(dl, desc='Evaluating', leave=False):
        xs = xs.to(device)
        preds = torch.argmax(model(xs), dim=1)
        gts.append(ys)
        predictions.append(preds.detach().cpu().numpy())
        
    return (accuracy_score(np.hstack(gts), np.hstack(predictions)),
            classification_report(np.hstack(gts), np.hstack(predictions), zero_division=0, digits=3))

In [ ]:
model = get_model('resnet50', weights='DEFAULT')
model.fc = nn.Linear(2048, 43)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

config = OmegaConf.create({
    'batch_size': 1024,
    'epochs': 8,
    'lr': 0.0001,
    'num_workers_dl': 8,
    'wd': 0.05
})

In [ ]:
dl_train = DataLoader(training_set, batch_size=config.batch_size, shuffle=True, num_workers=config.num_workers_dl, pin_memory=True)
dl_val = DataLoader(validation_set, batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers_dl, pin_memory=True)
dl_test = DataLoader(test_set, batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers_dl, pin_memory=True)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.wd)

losses = []
for epoch in tqdm(range(config.epochs)):
    train_loss = train_epoch(model, dl_train, optimizer, epoch+1, device=device)
    losses.append(train_loss)
    val_accuracy, cls_report = evaluate_model(model, dl_val, device=device)

    print(f'Epoch {epoch}: Training Loss: {train_loss}  Validation accuracy: {val_accuracy}')

plt.plot(losses)
(accuracy, cls_report) = evaluate_model(model, dl_test, device=device)
print(cls_report)

---
---
## Exercise 2: Pipeline Consolidation

Consolidate your implementation. When building applications based asd on Deep Learning, you will inevitably need to run many, many experiments. So, it is *always* a good idea to engineer a reproducible pipeline that allows you to run (and re-run) experiments with different hyperparameters. In this exercise you should do exactly this: engineer a deep learning pipeline that encapsulates (at least) training and evaluation so that you can easily and reproducibly run multiple experiments and compare the results.

Some things to think about when engineering this pipeline:
- **Model and Loss (and maybe Optimizer) Abstraction**: An important variable in training deep models is the model (obviously), the loss (somewhat less obvious), and the optimizer (less obvious) used during training. Your pipeline should probably be able to adapt to these changing configurations.
- **Configuration management**: Instead of global variables specializing each cell, thread a configuration object through your code (or use a singleton). If you are planning to use an IDE and not this notebook for the lab, you might consider using a configuration management library like [OmegaConf](https://omegaconf.readthedocs.io/en/2.3_branch/) which will help instrument your code so that you can pass configuration overrides via command line arguments.
- **Logging**: You will invariably need to run and compare multiple experiments. Instrument your code for monitoring training. Good options are [Tensorboard](https://docs.pytorch.org/tutorials/recipes/recipes/tensorboard_with_pytorch.html) or [Weights and Biases (WandB)](https://wandb.ai/site/). My PhD students seem to overwhelmingly prefer Weights and Biases. The configuration parameters should be used to facilitate distinguishing and comparing multiple runs with multiple hyperparameters.


---
---
## Exercise 3: Choose your Own Adventure

As promised, you should choose **one** of the following exercises to work. Well, at *least* one. If you want to do them all, that is also OK! 


---
### Exercise 3.3: *Detecting* Traffic Signs (hardest)

In this exercise you will see if you can take your pretrained *classification* backbone and turn it into a *detector*. We will need another dataset for this -- one with *full image frames* instead of cropped sign images. Luckily, this dataset is available on [Hugging Face](https://huggingface.co/datasets/keremberke/german-traffic-sign-detection) (see below for how to access it).

For this exercise, you could:
1. Start from an available Faster-RCNN model from `torchvision` and *replace* the feature extraction backbone with your ResNet fine-tuned on GTRSB.
2. This *stitched-together* network should do an OK-ish job already of detecting at least *some* traffic signs. See how it performs on some of the images from the GTRSB Detection dataset. Implement a simple function to visualize detections superimposed over the original image. Compute some detection metrics (accuracy @ IoU=0.5, for example).
3. If you're feeling ambitious, you could also *fine-tune* the resulting model to improve detection performance.

**NOTE**: The pretrained Faster-RCNN models in `torchvision` are all based on ResNet-50, so this exercise will likely be much less painful if you use a ResNet-50 from the very start.

**Also Note**: To use the **GTRSB Detection Dataset** you should use the `datasets` package from Hugging Face (version 3.X):

     # If using uv.
     uv add datasets==3

or

     # If using Anaconda
     conda install -c conda-forge datasets==3

In [ ]:
ds = load_dataset("keremberke/german-traffic-sign-detection", name="full")

Nella successiva cella, inizialmente sposto la vecchia `ResNet50` pre-addestrata su CPU, per poi andare ad estrarne il backbone e sostituirlo a quello della `faster_rcnn`.

La scelta dei parametri `min_size` e `max_size` è dovuta ad esigenze di VRAM (GPU NVIDIA RTX 5070 - **12 GB** VRAM)

In [ ]:
model = model.to('cpu')
backbone = _resnet_fpn_extractor(backbone=model, trainable_layers=4)

anchor_generator = _default_anchorgen() # anchor della v2: dimensioni (32, 64, 128, 256, 512) e aspect ratio (0.5, 1.0, 2.0)
rpn_head = RPNHead( # data una feature map, predice oggetto sì-mp + correzione del box
    backbone.out_channels, # canali in input (256 sarebbe l'output della fpn)
    anchor_generator.num_anchors_per_location()[0], # quante sagome testare per ogni punto
    conv_depth=2
)
box_head = FastRCNNConvFCHead(
    (backbone.out_channels, 7, 7), # input: 256 canali, finestra 7x7 (sarebbe l'output del RoIAlign)
    [256, 256, 256, 256], # 4 layer conv da 256 canali
    [1024], # i 4 layer conv sono seguiti da un layer fc
    norm_layer= nn.BatchNorm2d
)

model_f_cnn = FasterRCNN(
    backbone, # backbone della resnet pre-trainata + fpn
    num_classes=43,
    rpn_anchor_generator=anchor_generator,
    rpn_head=rpn_head,
    box_head=box_head,
    min_size=600,
    max_size=800
)
model_f_cnn = model_f_cnn.to(device)

Nella cella successiva ho stampato un elemento del dataset, in particolare del training set (ma è indifferente), allo scopo di vedere come è rappresentato:

In [ ]:
print(ds['train'][0]) 

Come si osserva, è rappresentato come un dizionario contenente diversi elementi. La cosa che salta immediatamente all'occhio è che le coordinate delle bounding box sono informato $$(x\_min, y\_min, w, h)$$ e non in formato $$(x\_min, y\_min, x\_max, y\_max)$$ come si aspetta la `faster_rcnn`. E' necessario dunque effettuare una conversione di formato.

Di seguito definisco una `collate_fn` adeguata al caso in esame: cioè, per l'addestramento e l'evaluation sono necessarie solo le immagini, le bounding boxes e le labels. 

In particolare, da notare come effettua la conversione di formato per le bounding boxes.

In [ ]:
def collate_fn(batch) -> tuple:
    images = []
    targets = []
    for element in batch:
        image = tf.to_tensor(element['image'])
        images.append(image)

        if len(element['objects']['bbox']) > 0:
            boxes = torch.as_tensor(element['objects']['bbox'], dtype=torch.float32)
            boxes = box_convert(boxes, in_fmt='xywh', out_fmt='xyxy')
            labels = torch.as_tensor(element['objects']['category'], dtype=torch.int64)
        else:
            boxes = torch.zeros(size=(0,4), dtype=torch.float32)
            labels = torch.zeros(size=(0, ), dtype=torch.int64)

        targets.append({'boxes': boxes, 'labels': labels})

    
    return (images, targets)

Attraverso la libreria `OmegaConf`, definisco una configurazione per gli iperparametri.

In [ ]:
config = OmegaConf.create({
    'batch_size': 8,
    'epochs': 50,
    'lr': 0.0001,
    'num_workers_dl': 8,
})

labels_names = ['animals', 'construction', 'cycles crossing', 'danger', 'no entry', 'pedestrian crossing', 'school crossing', 'snow', 'stop', 'bend', 'bend left', 'bend right', 'give way', 'go left', 'go left or straight', 'go right', 'go right or straight', 'go straight', 'keep left', 'keep right', 'no overtaking', 'no overtaking -trucks-', 'no traffic both ways', 'no trucks', 'priority at next intersection', 'priority road', 'restriction ends', 'restriction ends -overtaking -trucks--', 'restriction ends -overtaking-', 'restriction ends 80', 'road narrows', 'roundabout', 'slippery road', 'speed limit 100', 'speed limit 120', 'speed limit 20', 'speed limit 30', 'speed limit 50', 'speed limit 60', 'speed limit 70', 'speed limit 80', 'traffic signal', 'uneven road']

Istanzio i DataLoader:

In [ ]:
train_dl = DataLoader(dataset=ds['train'], batch_size=config.batch_size, shuffle=True, num_workers=config.num_workers_dl, collate_fn=collate_fn)
val_dl = DataLoader(dataset=ds['validation'], batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers_dl, collate_fn=collate_fn)
test_dl = DataLoader(dataset=ds['test'], batch_size=config.batch_size, shuffle=False, num_workers=config.num_workers_dl, collate_fn=collate_fn)

Istanzio l'ottimizzatore:

In [ ]:
optimizer = torch.optim.Adam(model_f_cnn.parameters(), lr=config.lr)

Definisco 3 funzioni:

1) `show_preds` : serve a disegnare i bounding boxes predetti (con relative label) sulle immagini.
2) `train_epoch`: contiene il training loop per una singola epoca. Successivamente verrà chiamata `config.epochs` volte.
3) `evaluate_model`: fa l'evaluation del modello su validation set/test set, con tanto di chiamata a `show_preds` (limitatamente, per evitare eccessiva verbosità).

In [ ]:
def show_preds(image, prediction, labels_names, score_thresh=0.5):

    image = (image.detach().cpu() * 255).to(torch.uint8)
    boxes = prediction['boxes'].detach().cpu()
    labels = prediction['labels'].detach().cpu()
    scores = prediction['scores'].detach().cpu()

    keep = scores >= score_thresh

    boxes = boxes[keep]
    labels = labels[keep]
    scores = scores[keep]

    if len(boxes) == 0:
        plt.figure(figsize=(13, 8))
        plt.imshow(tf.to_pil_image(image))
        plt.axis('off')
        plt.show()
        return

    captions = [f'{labels_names[l.item()]}: {s:.2f}' for l, s in zip(labels, scores)]
    ann = draw_bounding_boxes(
        image,
        boxes=boxes,
        labels=captions,
        colors='red',
        width=3
    )
    plt.figure(figsize=(13, 8))
    plt.imshow(tf.to_pil_image(ann))
    plt.axis('off')
    plt.show()

def train_epoch(model, dl, opt, epoch='Unknown', device='cpu'):
    model.train()
    for (batch_images, batch_targets) in tqdm(dl, desc=f'Training epoch {epoch}', leave=True):
        batch_images = [img.to(device) for img in batch_images]
        batch_targets = [{key: value.to(device) for key, value in t.items()} for t in batch_targets]
        opt.zero_grad()
        loss_dict = model(batch_images, batch_targets)
        total_loss = sum(loss for loss in loss_dict.values())
        total_loss.backward()
        opt.step()
    return total_loss.item()

@torch.no_grad()
def evaluate_model(model, dl, device, num_img_to_show):
    model.eval()
    all_predictions, all_targets = [], []
    shown_images = 0

    for (batch_images, batch_targets) in tqdm(dl, desc='Evaluating', leave=False):
        batch_images = [img.to(device) for img in batch_images]
        preds = model(batch_images)

        for i, p in enumerate(preds):
   
            if shown_images <= num_img_to_show: 
                show_preds(batch_images[i], p, labels_names)

            all_predictions.append({
                'boxes': p['boxes'].cpu(),
                'labels': p['labels'].cpu(),
                'scores': p['scores'].cpu()
            })
            all_targets.append({
                'boxes': batch_targets[i]['boxes'].cpu(),
                'labels': batch_targets[i]['labels'].cpu(),
            })

        
    return all_predictions, all_targets

Lancio l'addestramento per N (=50) epoche e per ogni epoca mi salvo la loss complessiva:

In [ ]:
train_losses = []
for epoch in tqdm(range(config.epochs)):
    total_loss = train_epoch(model_f_cnn, train_dl, optimizer, epoch+1, device=device)
    print(f'Total train loss: {total_loss:.3f}')
    train_losses.append(total_loss)

Di seguito stampo il grafico della loss sul training set:

In [ ]:
plt.plot(train_losses, marker='^', linestyle='--')
plt.title('Train Loss - Faster R_CNN')
plt.grid(True)

Valuto le prestazioni del modello attraverso la mAP a varie soglie di IoU:

In [ ]:
predictions, targets = evaluate_model(model_f_cnn, val_dl, device, num_img_to_show=3)

metric = MeanAveragePrecision(
    iou_type='bbox',
    iou_thresholds=[0.5, 0.75]
)

metric.update(predictions, targets)
result = metric.compute()


print(f'mAP: {result["map"]}   mAP@50: {result["map_50"]}   mAP@75: {result["map_75"]}')

---
---